In [ ]:
import sys, pathlib
# ensure repo root is on sys.path so src.* imports work when nbclient starts a kernel
sys.path.insert(0, str(pathlib.Path('.').resolve().parents[2]))


In [ ]:
# Use the Fuse parser + lowerer directly (more robust for nbclient smoke tests)
try:
    from src.parser import fuse_parser
    from src.lowering import FuseLowerer
except Exception:
    import importlib
    parser_mod = importlib.import_module('parser')
    lowering_main = importlib.import_module('lowering.main')
    fuse_parser = getattr(parser_mod, 'fuse_parser')
    FuseLowerer = getattr(lowering_main, 'FuseLowerer')

src = '''@fuse 1.2
node main(x: f32[1]) -> f32[1] {
  return Add(x, x)
}
'''
ast = fuse_parser.parse(src)
fl = FuseLowerer()
m = fl.lower(ast)
print('lowered', isinstance(m, object))
_fuse_model = m


In [ ]:
import numpy as np
import onnxruntime as ort

m = _fuse_model
sess = ort.InferenceSession(m.SerializeToString(), providers=['CPUExecutionProvider'])
input_name = m.graph.input[0].name
res = sess.run(None, {input_name: np.array([2], dtype=np.float32)})
print(float(res[0][0]))
